## read docs

In [3]:
from pathlib import Path

PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"
DOCUMENTS_DIR = DATA_DIR / "documents"

In [4]:
from pathlib import Path

docs = []

for file in Path(DOCUMENTS_DIR).glob("*.txt"):
    text = file.read_text(encoding="utf-8")

    docs.append({
        "source": file.name,
        "text": text
    })

docs

[{'source': 'cardiology_guidelines.txt',
  'text': 'Normal adult heart rate ranges from 60 to 100 bpm.\nPersistent tachycardia should be evaluated.'},
 {'source': 'icu_protocol.txt',
  'text': 'Patients in ICU should maintain oxygen saturation above 94%.\nHeart rate above 100 bpm may indicate tachycardia.'},
 {'source': 'respiratory_guidelines.txt',
  'text': 'Oxygen saturation below 90% is considered critical.\nImmediate intervention may be required.'}]

## Chunking

In [5]:
chunks = []

chunk_size = 150

for doc in docs:

    text = doc["text"]

    for i in range(0, len(text), chunk_size):

        chunks.append({
            "source": doc["source"],
            "text": text[i:i + chunk_size]
        })

len(chunks)

3

## Embeddings

In [6]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

c:\Users\reems\Desktop\patient-monitoring-capstone\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [7]:
texts = [c["text"] for c in chunks]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True
)

embeddings.shape

(3, 384)

## Vector Store

In [8]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(
    np.array(embeddings)
)

print(index.ntotal)

3


## Dense Retrieval

In [9]:
def dense_search(
    query,
    top_k=3
):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for idx in indices[0]:
        results.append(chunks[idx])

    return results

In [10]:
dense_search(
    "dangerously low oxygen levels"
)

[{'source': 'respiratory_guidelines.txt',
  'text': 'Oxygen saturation below 90% is considered critical.\nImmediate intervention may be required.'},
 {'source': 'icu_protocol.txt',
  'text': 'Patients in ICU should maintain oxygen saturation above 94%.\nHeart rate above 100 bpm may indicate tachycardia.'},
 {'source': 'cardiology_guidelines.txt',
  'text': 'Normal adult heart rate ranges from 60 to 100 bpm.\nPersistent tachycardia should be evaluated.'}]

## BM25

In [11]:
from rank_bm25 import BM25Okapi

tokenized_chunks = [
    chunk["text"].lower().split()
    for chunk in chunks
]

bm25 = BM25Okapi(
    tokenized_chunks
)


def bm25_search(
    query,
    top_k=3
):
    scores = bm25.get_scores(
        query.lower().split()
    )

    ranked = sorted(
        enumerate(scores),
        key=lambda x: x[1],
        reverse=True
    )

    results = []

    for idx, score in ranked[:top_k]:
        results.append(chunks[idx])

    return results


bm25_search(
    "heart rate"
)

[{'source': 'cardiology_guidelines.txt',
  'text': 'Normal adult heart rate ranges from 60 to 100 bpm.\nPersistent tachycardia should be evaluated.'},
 {'source': 'icu_protocol.txt',
  'text': 'Patients in ICU should maintain oxygen saturation above 94%.\nHeart rate above 100 bpm may indicate tachycardia.'},
 {'source': 'respiratory_guidelines.txt',
  'text': 'Oxygen saturation below 90% is considered critical.\nImmediate intervention may be required.'}]

## Hybrid Search

In [12]:
def hybrid_search(
    query,
    top_k=5
):
    dense_results = dense_search(
        query,
        top_k
    )

    bm25_results = bm25_search(
        query,
        top_k
    )

    combined = []

    seen = set()

    for result in dense_results + bm25_results:

        key = (
            result["source"],
            result["text"]
        )

        if key not in seen:

            combined.append(result)
            seen.add(key)

    return combined[:top_k]


hybrid_search(
    "tachycardia"
)

[{'source': 'cardiology_guidelines.txt',
  'text': 'Normal adult heart rate ranges from 60 to 100 bpm.\nPersistent tachycardia should be evaluated.'},
 {'source': 'icu_protocol.txt',
  'text': 'Patients in ICU should maintain oxygen saturation above 94%.\nHeart rate above 100 bpm may indicate tachycardia.'},
 {'source': 'respiratory_guidelines.txt',
  'text': 'Oxygen saturation below 90% is considered critical.\nImmediate intervention may be required.'}]

## Reciprocal Rank Fusion (RRF)

In [13]:
def dense_search_with_indices(query, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    return indices[0]


def bm25_search_with_indices(query, top_k=5):

    scores = bm25.get_scores(
        query.lower().split()
    )

    ranked = sorted(
        enumerate(scores),
        key=lambda x: x[1],
        reverse=True
    )

    return [idx for idx, _ in ranked[:top_k]]



def rrf_search(query, top_k=5, k=60):

    dense_rank = dense_search_with_indices(
        query,
        top_k
    )

    bm25_rank = bm25_search_with_indices(
        query,
        top_k
    )

    scores = {}

    for rank, doc_id in enumerate(dense_rank):

        scores[doc_id] = (
            scores.get(doc_id, 0)
            + 1 / (k + rank + 1)
        )

    for rank, doc_id in enumerate(bm25_rank):

        scores[doc_id] = (
            scores.get(doc_id, 0)
            + 1 / (k + rank + 1)
        )

    ranked_docs = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    results = []

    for doc_id, score in ranked_docs[:top_k]:

        results.append(
            {
                "score": score,
                "source": chunks[doc_id]["source"],
                "text": chunks[doc_id]["text"]
            }
        )

    return results


rrf_search(
    "tachycardia"
)

[{'score': 0.03278688524590164,
  'source': 'cardiology_guidelines.txt',
  'text': 'Normal adult heart rate ranges from 60 to 100 bpm.\nPersistent tachycardia should be evaluated.'},
 {'score': 0.03225806451612903,
  'source': 'icu_protocol.txt',
  'text': 'Patients in ICU should maintain oxygen saturation above 94%.\nHeart rate above 100 bpm may indicate tachycardia.'},
 {'score': 0.031746031746031744,
  'source': 'respiratory_guidelines.txt',
  'text': 'Oxygen saturation below 90% is considered critical.\nImmediate intervention may be required.'},
 {'score': 0.031009615384615385,
  'source': 'respiratory_guidelines.txt',
  'text': 'Oxygen saturation below 90% is considered critical.\nImmediate intervention may be required.'}]

## Cross Encoder Reranking

In [14]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)



def rerank(query, retrieved_docs):

    pairs = [
        (query, doc["text"])
        for doc in retrieved_docs
    ]

    scores = reranker.predict(
        pairs
    )

    ranked = sorted(
        zip(retrieved_docs, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked



results = rrf_search(
    "What is tachycardia?"
)

reranked = rerank(
    "What is tachycardia?",
    results
)

reranked

c:\Users\reems\Desktop\patient-monitoring-capstone\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[({'score': 0.03252247488101534,
   'source': 'cardiology_guidelines.txt',
   'text': 'Normal adult heart rate ranges from 60 to 100 bpm.\nPersistent tachycardia should be evaluated.'},
  np.float32(-0.30904466)),
 ({'score': 0.03200204813108039,
   'source': 'icu_protocol.txt',
   'text': 'Patients in ICU should maintain oxygen saturation above 94%.\nHeart rate above 100 bpm may indicate tachycardia.'},
  np.float32(-1.7845435)),
 ({'score': 0.032266458495966696,
   'source': 'respiratory_guidelines.txt',
   'text': 'Oxygen saturation below 90% is considered critical.\nImmediate intervention may be required.'},
  np.float32(-11.366825)),
 ({'score': 0.031009615384615385,
   'source': 'respiratory_guidelines.txt',
   'text': 'Oxygen saturation below 90% is considered critical.\nImmediate intervention may be required.'},
  np.float32(-11.366825))]

## Answer Generation using FLAN-T5

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)



In [ ]:

def rag_answer(query):

    retrieved = rrf_search(
        query,
        top_k=5
    )

    reranked = rerank(
        query,
        retrieved
    )

    top_docs = [
        doc
        for doc, score in reranked[:3]
    ]

    context = "\n".join(
        doc["text"]
        for doc in top_docs
    )

    prompt = f"""
You are a medical assistant.
Answer the question using ONLY the context provided.
Give a complete sentence and include the source information.

Context:
{context}

Question:
{query}

Answer:
"""

    response = generator(
    prompt,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.3
)

    return {
        "question": query,
        "answer": response[0]["generated_text"],
        "citations": list(
            set(
                doc["source"]
                for doc in top_docs
            )
        )
    }


rag_answer(
    "What oxygen level is considered critical?"
)

{'question': 'What oxygen level is considered critical?',
 'answer': 'Oxygen saturation below 90%',
 'citations': ['respiratory_guidelines.txt', 'icu_protocol.txt']}

##  gen Answer + Citations

In [27]:
def answer_with_citations(query):

    retrieved = rrf_search(
        query,
        top_k=5
    )

    reranked = rerank(
        query,
        retrieved
    )

    top_docs = [
        doc
        for doc, score in reranked[:3]
    ]

    answer_parts = []

    citations = []

    for doc in top_docs:

        answer_parts.append(
            doc["text"]
        )

        citations.append(
            doc["source"]
        )

    return {
        "question": query,
        "answer": " ".join(answer_parts),
        "citations": citations
    }




In [28]:
result = answer_with_citations(
    "What is tachycardia?"
)

result

{'question': 'What is tachycardia?',
 'answer': 'Normal adult heart rate ranges from 60 to 100 bpm.\nPersistent tachycardia should be evaluated. Patients in ICU should maintain oxygen saturation above 94%.\nHeart rate above 100 bpm may indicate tachycardia. Oxygen saturation below 90% is considered critical.\nImmediate intervention may be required.',
 'citations': ['cardiology_guidelines.txt',
  'icu_protocol.txt',
  'respiratory_guidelines.txt']}

In [29]:
answer_with_citations(
    "What oxygen level is critical?"
)

{'question': 'What oxygen level is critical?',
 'answer': 'Oxygen saturation below 90% is considered critical.\nImmediate intervention may be required. Oxygen saturation below 90% is considered critical.\nImmediate intervention may be required. Patients in ICU should maintain oxygen saturation above 94%.\nHeart rate above 100 bpm may indicate tachycardia.',
 'citations': ['respiratory_guidelines.txt',
  'respiratory_guidelines.txt',
  'icu_protocol.txt']}

In [30]:
answer_with_citations(
    "What heart rate is considered normal?"
)

{'question': 'What heart rate is considered normal?',
 'answer': 'Normal adult heart rate ranges from 60 to 100 bpm.\nPersistent tachycardia should be evaluated. Patients in ICU should maintain oxygen saturation above 94%.\nHeart rate above 100 bpm may indicate tachycardia. Oxygen saturation below 90% is considered critical.\nImmediate intervention may be required.',
 'citations': ['cardiology_guidelines.txt',
  'icu_protocol.txt',
  'respiratory_guidelines.txt']}